In [1]:
import torch
from torch.profiler import profile, ProfilerActivity, record_function

# Model Initialization Testing

In [8]:
# See if model can be initialized
from kdv import *

INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = 42, 
    verbose                  = True,
)
model = KDV(INIT_PARAMS)

Using device: cuda


# Loss Function Testing

First step is testing if these functions both run and output the expected results

In [3]:
from kdv_loss import *
from kdv_trainer import *

In [4]:
weights = {
    'w_ic': 10.7,
    'w_bc': 2.0
    #'w_pde': 20.0 #missing on purpose to test update works correctly
}

domain = setup_training_domain(1000, 100, 100, model.soliton_params)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Test loss dict
losses = init_loss_list()
print(f'The loss dict: {losses}')

#Test loss weights
loss_weights = init_loss_weights(device, init_weights=None)
print(f'The loss wieghts as default: {loss_weights}')
loss_weights = init_loss_weights(device, weights)
print(f'The loss wieghts with dict: {loss_weights}')
print(f'Loss weights device: {loss_weights.device} \n')

#loss_comps = torch.tensor([0.3, 4.3, 1.0], device=loss_weights.device) #dummy var, check loss_components function with proper neural network
loss_comps = loss_components(model.neural_net, domain=domain)
total_loss = compute_total_loss(loss_weights, loss_comps)
#make sure that the dot operator works as intended
print(f'Total loss: {total_loss}')

The loss dict: {'total': [], 'initial': [], 'boundary': [], 'pde': []}
The loss wieghts as default: tensor([1., 1., 1.], device='cuda:0')
The loss wieghts with dict: tensor([10.7000,  2.0000,  1.0000], device='cuda:0')
Loss weights device: cuda:0 

Total loss: 4.241991996765137


/home/jairdan/miniconda3/envs/soliton-pinns/lib/python3.14/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


# Train Function Testing

In [9]:
#See if model can be trained
TRAIN_PARAMS = dict(
    adam_epochs              = 1000,
    verbose_step             = 100,
    n_collocation            = 50000, 
    n_initial                = 30000,  
    n_boundary               = 10000,    
    adam_lr                  = 1e-3,   
    lbfgs_lr                 = 1.0,    
    lbfgs_history_size       = 100,  
    lbfgs_version            = 'old',
    adaptive_sampling        = False,   
)

TRAIN_WEIGHTS = dict(
    w_ic                     = 5.0,    
    w_bc                     = 1.0,    
    w_pde                    = 15.0,
)

training_stats = model.train(TRAIN_PARAMS, TRAIN_WEIGHTS)

Starting Adam optimization...
[gpu mem] train start               alloc    23.4 MB  reserved    52.0 MB  peak    23.4 MB
Adam - Epoch 0/1000, Total Loss: 2.250976e+00
Adam - Epoch 100/1000, Total Loss: 1.023836e-02
Adam - Epoch 200/1000, Total Loss: 5.153835e-03
Adam - Epoch 300/1000, Total Loss: 2.497639e-03
Adam - Epoch 400/1000, Total Loss: 1.313612e-03
Adam - Epoch 500/1000, Total Loss: 7.931188e-04
Adam - Epoch 600/1000, Total Loss: 5.365211e-04
Adam - Epoch 700/1000, Total Loss: 3.933690e-04
Adam - Epoch 800/1000, Total Loss: 3.077947e-04
Adam - Epoch 900/1000, Total Loss: 2.531004e-04
Adam - Epoch 999/1000, Total Loss: 2.158611e-04
[gpu mem] after Adam                alloc    24.2 MB  reserved   834.0 MB  peak   748.3 MB

Starting L-BFGS optimization...
L-BFGS - Iteration 100, Total Loss: 4.875926e-05
L-BFGS - Iteration 200, Total Loss: 2.254033e-05
L-BFGS - Iteration 300, Total Loss: 1.185265e-05
L-BFGS - Iteration 400, Total Loss: 6.703158e-06
L-BFGS - Iteration 500, Total Los

# Testing Function Testing

In [13]:
model.test(1000, 1000, 'absolute-normalized')

absolute-normalized error metrics:
Mean: 2.032103e-03
Maximum: 3.230447e-02


True

# Loss Function Profiling

In [ ]:
def test_loss_run():
    #add the sequence of loss function calls you want to run here
    
    return

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], 
    record_shapes=True, 
    profile_memory=True
    with_modules=True
) as p:
    test_loss_run()

print('Non-compile: \n')
print(p.key_averages().table(sort_by='self_cuda_time_total', row_limit=-1))

torch.compile(test_loss_run)
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], 
    record_shapes=True, 
    profile_memory=True
    with_modules=True
) as p:
    test_loss_run()

print('Compile: \n')
print(p.key_averages().table(sort_by='self_cuda_time_total', row_limit=-1))

